# Import

Import all necessary libraries. If you face any issue, please install the libraries using pip or conda.

Ensure reproducibility of your code and keep reusing your functions in all assignments.

**Note: NetworkX must not be used in your solution except explicitly mentioned in the questions, and it is only for checking your results.**

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import sys

In [2]:
# change this to the filename you want to read
undirected_network_edges = "undirected_network_edge_list.csv"
directed_network_edges = "directed_network_edge_list.csv"

# don't change this part
if len(sys.argv) > 2 and 'ipykernel' not in sys.modules:
    print("Running in shell, reading filename from command line argument")
    undirected_network_filename = sys.argv[1]
    directed_network_filename = sys.argv[2]

# Getting started

Paste all your previous functions here. This will become your library of functions that you can reuse in all assignments.

In [3]:
def read_network(filename, directed=False):
    pass
    # read edge list into df
    df = pd.read_csv(undirected_network_edges)
    
    # sort the node IDs into node_list
    all_nodes = set(df['from']) | set(df['to'])
    node_list = np.array(sorted(list(all_nodes)))
    
    # convert df into adjacency matrix adj ordered by node_list
    n = len(node_list)
    adj = np.zeros((n, n), dtype=int)
    
    node_to_index = {node: i for i, node in enumerate(node_list)}
    
    # Fill the adjacency matrix
    for _, row in df.iterrows():
        source = row['from']
        target = row['to']
        i = node_to_index[source]
        j = node_to_index[target]
        adj[i, j] = 1
        
        # If undirected, also set the symmetric entry
        if not directed:
            adj[j, i] = 1
    
    # read the edge list using NetworkX into G
    if directed:
        G = nx.from_pandas_edgelist(df, source='from', target='to', edge_attr='weight', 
                                   create_using=nx.DiGraph())
    else:
        G = nx.from_pandas_edgelist(df, source='from', target='to', edge_attr='weight')
    
    return adj, node_list, G

In [4]:
adj, node_list, G = read_network(undirected_network_edges, directed=False)

In [5]:
def clustering_coefficient(adj, is_directed=False):
    pass
    # Step 1: Symmetrize if directed (treat as undirected for clustering)
    if is_directed:
        adj_sym = (adj + adj.T > 0).astype(int)  # Union of edges in both directions
    else:
        adj_sym = adj
    
    n = adj_sym.shape[0]  # number of nodes
    clustering_coeffs = np.zeros(n)
    
    # Step 2: For each node, compute its clustering coefficient
    for i in range(n):
        # Find neighbors of node i
        neighbors = np.where(adj_sym[i] == 1)[0]
        k_i = len(neighbors)  # degree of node i
        
        # If node has 0 or 1 neighbors, clustering coefficient is 0 (undefined, set to 0)
        if k_i < 2:
            clustering_coeffs[i] = 0.0
            continue
        
        # Step 3: Count edges between neighbors
        # Extract the subgraph of neighbors
        edges_between_neighbors = 0
        for idx1 in range(len(neighbors)):
            for idx2 in range(idx1 + 1, len(neighbors)):
                neighbor1 = neighbors[idx1]
                neighbor2 = neighbors[idx2]
                # Check if there's an edge between neighbor1 and neighbor2
                if adj_sym[neighbor1, neighbor2] == 1:
                    edges_between_neighbors += 1
        
        # Step 4: Compute clustering coefficient
        # Maximum possible edges = k_i * (k_i - 1) / 2
        max_possible_edges = k_i * (k_i - 1) / 2
        clustering_coeffs[i] = edges_between_neighbors / max_possible_edges
    
    # Step 5: Compute average clustering coefficient
    average_clustering_coeff = np.mean(clustering_coeffs)
    
    return clustering_coeffs, average_clustering_coeff

In [6]:
def largest_connected_component(adj):
    pass
    n = adj.shape[0]  # number of nodes
    visited = np.zeros(n, dtype=bool)  # track visited nodes
    components = []  # store all connected components
    
    # Helper function: BFS to find one connected component starting from node 'start'
    def bfs(start):
        component = set()
        queue = [start]
        visited[start] = True
        
        while queue:
            node = queue.pop(0)
            component.add(node)
            
            # Find all neighbors of current node
            neighbors = np.where(adj[node] == 1)[0]
            
            for neighbor in neighbors:
                if not visited[neighbor]:
                    visited[neighbor] = True
                    queue.append(neighbor)
        
        return component
    
    # Find all connected components
    for node in range(n):
        if not visited[node]:
            component = bfs(node)
            components.append(component)
    
    # Find the largest connected component (in terms of indices)
    lcc_indices = max(components, key=len)
    
    # Map indices back to original node IDs using node_list
    lcc_node_list = {node_list[idx] for idx in lcc_indices}
    
    return lcc_node_list

In [7]:
def ER_graph_generator(n, p, seed=0):
    pass
    # Step 1: Fix the random seed
    rng = np.random.Generator(np.random.PCG64(seed))
    
    # Step 2: Generate the graph
    # Create random matrix where each entry is 1 with probability p
    adj = (rng.random((n, n)) < p).astype(int)
    
    # Step 3: Remove self-loops (set diagonal to 0)
    np.fill_diagonal(adj, 0)
    
    # Step 4: Make the graph undirected
    # Use upper triangular part and mirror it to lower triangular part
    adj = np.triu(adj)  # Keep only upper triangular part
    adj = adj + adj.T   # Add transpose to make symmetric
    
    # Step 5: Return the adjacency matrix
    return adj

In [8]:
def degree(adj):
    pass
    degrees = np.sum(adj, axis=1)
    return degrees

In [9]:
def Configuration_model_generator(deg_seq, seed=0):
    pass
     # Step 1: Fix the random seed
    rng = np.random.Generator(np.random.PCG64(seed))
    
    # Step 2: Initialize the graph with no edges
    n = len(deg_seq)
    adj = np.zeros((n, n), dtype=int)
    
    # Step 3: Create a list of stubs (half-edges)
    # Each node i appears deg_seq[i] times in the stub list
    stubs = []
    for node, degree in enumerate(deg_seq):
        stubs.extend([node] * degree)
    
    # Convert to numpy array for easier manipulation
    stubs = np.array(stubs)
    
    # Step 4: Randomly shuffle the stubs
    rng.shuffle(stubs)
    
    # Step 5: Pair up stubs to create edges
    # Take pairs of stubs and connect them
    for i in range(0, len(stubs) - 1, 2):
        node1 = stubs[i]
        node2 = stubs[i + 1]
        
        # Add edge (allows self-loops and multi-edges)
        adj[node1, node2] += 1
        adj[node2, node1] += 1
    
    # Step 6: Return the adjacency matrix
    return adj

In [10]:
def WS_graph_generator(n, k, p, seed=0):
    """
    Generate an undirected WS random graph with n nodes, starting with degree k and rewiring probability p.
    Your graph cannot have self-loops or multiple edges.
    Args:
        n: number of nodes (int)
        k: initial average degree (int, must be even)
        p: probability of rewiring (float)
        seed: random seed (int)
    Returns:
        adj: adjacency matrix of the generated WS graph (numpy array)
    """

    rng = np.random.Generator(np.random.PCG64(seed))
    pass
    # Step 1: Start with a ring lattice
    # Initialize adjacency matrix
    adj = np.zeros((n, n), dtype=int)
    
    # Connect each node to its k/2 nearest neighbors on each side
    for i in range(n):
        for j in range(1, k // 2 + 1):
            neighbor = (i + j) % n  # Wrap around using modulo
            adj[i, neighbor] = 1
            adj[neighbor, i] = 1
    
    # Step 2: Save the edges (we'll iterate through half of them to avoid duplication)
    # We only need to consider edges where i < j to avoid processing each edge twice
    edges_to_rewire = []
    for i in range(n):
        for j in range(i + 1, n):
            if adj[i, j] == 1:
                edges_to_rewire.append((i, j))
    
    # Step 3: Rewire edges with probability p
    for i, j in edges_to_rewire:
        # Rewire with probability p
        if rng.random() < p:
            # Remove the old edge
            adj[i, j] = 0
            adj[j, i] = 0
            
            # Step 4: Find a new target node (keep source node i)
            # Try to connect i to a new node that isn't i itself and doesn't already have an edge
            max_attempts = 100  # Prevent infinite loops
            rewired = False
            
            for _ in range(max_attempts):
                # Pick a random target node
                new_target = rng.integers(0, n)
                
                # Check if valid: not a self-loop and edge doesn't already exist
                if new_target != i and adj[i, new_target] == 0:
                    # Add the new edge
                    adj[i, new_target] = 1
                    adj[new_target, i] = 1
                    rewired = True
                    break
            
            # If we couldn't rewire after many attempts, keep the original edge
            if not rewired:
                adj[i, j] = 1
                adj[j, i] = 1
    
    # Step 5: Return the adjacency matrix
    return adj

In [11]:
def BA_random_graph_generator(m0, m, n, seed=0):
    """
    Generate an undirected BA random graph with n nodes, starting with m0 nodes and adding m edges for each new node.
    Args:
        m0: initial number of nodes (int)
        m: number of edges to attach from the new node to existing nodes (int)
        n: total number of nodes in the graph (int)
        seed: random seed (int)
    Returns:
        adj: adjacency matrix of the generated BA graph (numpy array)
    """

    assert m0>=m, "m0 must be greater than or equal to m"
    rng = np.random.Generator(np.random.PCG64(seed))

    pass
    # Step 1: Start with a small fully-connected network of m0 nodes
    adj = np.zeros((n, n), dtype=int)
    
    # Create fully connected initial network
    for i in range(m0):
        for j in range(i + 1, m0):
            adj[i, j] = 1
            adj[j, i] = 1
    
    # Step 2 & 3: Add new nodes one by one
    for new_node in range(m0, n):
        # Get current degrees of all existing nodes
        degrees = np.sum(adj[:new_node, :new_node], axis=1)
        
        # Calculate probabilities proportional to degree
        # Probability that new node connects to node i is proportional to degree of i
        total_degree = np.sum(degrees)
        
        if total_degree == 0:
            # If all degrees are 0 (shouldn't happen with m0 fully connected), use uniform
            probabilities = np.ones(new_node) / new_node
        else:
            probabilities = degrees / total_degree
        
        # Select m different nodes to connect to (without replacement)
        # Use preferential attachment based on degree
        targets = rng.choice(new_node, size=m, replace=False, p=probabilities)
        
        # Add edges from new_node to selected targets
        for target in targets:
            adj[new_node, target] = 1
            adj[target, new_node] = 1
    
    return adj

In [12]:
def average_path_distance(adj):
    """
    Compute the average shortest path distance of the largest connected component of the provided undirected network.
    Note that this function only work for the provided undirected network data.
    Note that you are allowed to use NetworkX for this function.
    Args:
        adj: adjacency matrix (numpy array)
    Returns:
        avg_path_length: distance (float)
    """
    
    pass
    # Step 1: Extract the largest connected component (returns indices, not node IDs)
    lcc_indices = largest_connected_component(adj)
    
    # Convert set to sorted list
    lcc_indices_list = sorted(list(lcc_indices))
    
    # Step 2: Extract subgraph adjacency matrix using np.ix_
    lcc_adj = adj[np.ix_(lcc_indices_list, lcc_indices_list)]
    
    # Step 3: Compute average shortest path using NetworkX
    G_lcc = nx.from_numpy_array(lcc_adj)
    avg_path_length = nx.average_shortest_path_length(G_lcc)
    
    return avg_path_length

# Question 1 (5 points)

Design a function to compute the farness, closeness and harmonic centrality of all nodes in a given undirected graph.

1) Find the largest connected component of the graph.
2) compute pairwise shortest path lengths between all nodes in the largest connected component using `nx.shortest_path_length`.
3) Using the pairwise shortest path lengths, compute farness, closeness and harmonic centrality of all nodes in the largest connected component.

In [13]:
def average_distance(adj, method="farness"):
    """
    Compute the average distance of nodes in a graph.
    Args:
        adj: adjacency matrix (numpy array)
        method: type of distance metric ("farness", "closeness", or "harmonic")
    Returns:
        distances: dict of average distances for each node (dict of float)
    """
    pass
    # Step 1: Find the largest connected component
    lcc_indices = largest_connected_component(adj)
    lcc_indices_list = sorted(list(lcc_indices))
    
    # Extract subgraph adjacency matrix for LCC
    lcc_adj = adj[np.ix_(lcc_indices_list, lcc_indices_list)]
    
    # Convert to NetworkX graph
    G_lcc = nx.from_numpy_array(lcc_adj)
    
    # Step 2: Compute pairwise shortest path lengths
    # nx.shortest_path_length returns a dict of dicts: {source: {target: length}}
    all_shortest_paths = dict(nx.shortest_path_length(G_lcc))
    
    # Step 3: Compute the requested centrality metric
    distances = {}
    n = len(lcc_indices_list)  # number of nodes in LCC
    
    for node in range(n):
        # Get shortest paths from this node to all others
        paths_from_node = all_shortest_paths[node]
        
        if method == "farness":
            # Farness: sum of all shortest path distances from node to all others
            # F(v) = Σ d(v,u) for all u ≠ v
            farness = sum(paths_from_node.values())
            distances[node] = farness
            
        elif method == "closeness":
            # Closeness: reciprocal of farness (normalized)
            # C(v) = (n-1) / Σ d(v,u)
            farness = sum(paths_from_node.values())
            if farness > 0:
                closeness = (n - 1) / farness
            else:
                closeness = 0.0
            distances[node] = closeness
            
        elif method == "harmonic":
            # Harmonic centrality: sum of reciprocals of distances
            # H(v) = Σ 1/d(v,u) for all u ≠ v (excluding u=v where d=0)
            harmonic = sum(1.0 / d for d in paths_from_node.values() if d > 0)
            distances[node] = harmonic
            
        else:
            raise ValueError(f"Unknown method: {method}. Choose 'farness', 'closeness', or 'harmonic'")
    
    return distances

# Question 2 (5 points)

Design a function to compute the normalized betweenness of an undirected network.

1) Compute all pairwise shortest paths between all nodes in the graph using `nx.all_pairs_all_shortest_paths`.
2) For each node, compute the fraction of shortest paths that pass through the node.
3) Normalize the betweenness scores by dividing by the total number of shortest paths.

In [16]:
def betweenness(G):
    """
    Compute the normalized betweenness of an undirected network.
    Args:
        G: NetworkX graph
    Returns:
        betweenness: dict of normalized betweenness for each node (dict of float)
    """
    # Initialize betweenness scores
    betweenness_scores = {node: 0.0 for node in G.nodes()}
    
    # Step 1: Compute all pairwise shortest paths
    all_shortest_paths = dict(nx.all_pairs_all_shortest_paths(G))
    
    # Step 2: Count shortest paths passing through each node
    for source in G.nodes():
        for target in G.nodes():
            if source >= target:  # For undirected graph, only count each pair once
                continue
            
            # Get all shortest paths from source to target
            if target in all_shortest_paths[source]:
                paths = all_shortest_paths[source][target]
                num_paths = len(paths)
                
                # For each shortest path, credit intermediate nodes
                for path in paths:
                    for node in path[1:-1]:  # Exclude source and target
                        betweenness_scores[node] += 1.0 / num_paths
    
    # Step 3: Normalize by (n-1)(n-2)/2 for undirected graphs
    n = len(G.nodes())
    if n > 2:
        normalization = (n - 1) * (n - 2) / 2.0
        for node in betweenness_scores:
            betweenness_scores[node] /= normalization
    
    return betweenness_scores

Check your results using NetworkX's betweenness function.

In [17]:
assert np.allclose(pd.Series(betweenness(G)).sort_index().values,
                   pd.Series(nx.betweenness_centrality(G, normalized=True)).sort_index().values), "Question 2 is incorrect"
print("Question 2 is correct")

Question 2 is correct


# Question 3 (10 points)
Implement the PageRank algorithm for undirected graphs on slide 51.

1) Find the largest connected component of the graph.
2) Initialize the PageRank score of all nodes randomly such that the sum of all PageRank scores is 1.
3) Update the PageRank scores iteratively using the formula on slide 51
4) Normalize the PageRank scores after each iteration such that the sum of all PageRank scores is 1.
5) Stop the iterations after a fixed number of iterations

In [25]:
def pagerank(adj, alpha, iteration=100, seed=0):
    """
    Compute the Pagerank scores of LCC nodes in a graph using the iteration method.
    Args:
        adj: adjacency matrix (numpy array)
        alpha: damping factor (float)
        iteration: number of iterations (int)
        seed: random seed (int)
    Returns:
        pagerank_scores: dict of Pagerank scores for each LCC node (dict of float)
    """
    rng = np.random.Generator(np.random.PCG64(seed))
    
    # Step 1: Find the largest connected component
    # This returns a set of node IDs from node_list
    lcc_node_ids = largest_connected_component(adj)
    
    # Create mapping from node_list IDs to adjacency matrix indices
    node_id_to_idx = {node_id: idx for idx, node_id in enumerate(node_list)}
    lcc_indices = sorted([node_id_to_idx[node_id] for node_id in lcc_node_ids])
    n = len(lcc_indices)
    
    # Extract subgraph adjacency matrix for LCC
    lcc_adj = adj[np.ix_(lcc_indices, lcc_indices)]
    
    # Step 2: Initialize PageRank scores randomly (sum = 1)
    pr = rng.random(n)
    pr = pr / pr.sum()
    
    # Compute degree of each node
    degrees = np.sum(lcc_adj, axis=1).astype(float)
    degrees[degrees == 0] = 1  # Avoid division by zero
    
    # Create transition matrix: M[i,j] = adj[i,j] / degree[j]
    M = lcc_adj.T / degrees  # Transpose and normalize by out-degree
    
    # Personalization vector (uniform)
    personalization = np.ones(n) / n
    
    # Step 3 & 4: Update PageRank scores iteratively
    for _ in range(iteration):
        pr_new = (1 - alpha) * personalization + alpha * (M @ pr)
        pr = pr_new / pr_new.sum()
    
    # Step 5: Return as dictionary with original node IDs from node_list
    pagerank_scores = {}
    for i, adj_idx in enumerate(lcc_indices):
        original_node_id = node_list[adj_idx]
        pagerank_scores[original_node_id] = pr[i]
    
    return pagerank_scores

Check your results using NetworkX's pagerank function.

In [26]:
lcc_node_list = list(next(nx.connected_components(G)))

assert np.allclose(
            pd.Series(pagerank(adj, alpha=0.85, iteration=100, seed=0)).sort_index().values,
            pd.Series(nx.pagerank(
                        G.subgraph(lcc_node_list), 
                        alpha=0.85, 
                        personalization=dict(zip(lcc_node_list, np.ones(len(lcc_node_list))))
                    )).sort_index().values,
            atol=1e-3
        ), "Your pagerank function deviates from NetworkX's pagerank function by more than 1e-3!"
print("Your pagerank function passed the test!")

Your pagerank function passed the test!
